##Transform Races Data

1. Read bronze races table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake_case (circuitId->circuit_id, raceName->race_name)
4. Rename columns to make them more meaningful (date->race_date)
5. Remove duplicate records
6. Transform values of columns race_name to Title Case
7. Write the transformed data to silver races table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

####Step 1: Read the bronze races table

In [0]:
races_df = (
    spark.table(bronze_table)
        .filter((F.col("batch_id") == v_batch_id))
)

In [0]:
display(races_df)

season,round,url,raceName,date,circuitId,ingestion_timestamp,source_file,batch_id
1950,1,https://en.wikipedia.org/wiki/1950_British_Grand_Prix,british grand prix,1950-05-13,silverstone,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,2,https://en.wikipedia.org/wiki/1950_Monaco_Grand_Prix,monaco grand prix,1950-05-21,monaco,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,3,https://en.wikipedia.org/wiki/1950_Indianapolis_500,indianapolis 500,1950-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,4,https://en.wikipedia.org/wiki/1950_Swiss_Grand_Prix,swiss grand prix,1950-06-04,bremgarten,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,5,https://en.wikipedia.org/wiki/1950_Belgian_Grand_Prix,belgian grand prix,1950-06-18,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,6,https://en.wikipedia.org/wiki/1950_French_Grand_Prix,french grand prix,1950-07-02,reims,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,7,https://en.wikipedia.org/wiki/1950_Italian_Grand_Prix,italian grand prix,1950-09-03,monza,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,1,https://en.wikipedia.org/wiki/1951_Swiss_Grand_Prix,swiss grand prix,1951-05-27,bremgarten,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,2,https://en.wikipedia.org/wiki/1951_Indianapolis_500,indianapolis 500,1951-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,3,https://en.wikipedia.org/wiki/1951_Belgian_Grand_Prix,belgian grand prix,1951-06-17,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01


####Step 2: Keep only the coluumns required for analysis (Drop url column)

There are 2 ways to do this... one by selecting the required columns and the other by dropping the unwanted columns

In [0]:
##This method uses selection of the required columns
from pyspark.sql import functions as F

races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

display(races_selected_df)

season,round,raceName,date,circuitId,ingestion_timestamp,source_file,batch_id
1950,1,british grand prix,1950-05-13,silverstone,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,2,monaco grand prix,1950-05-21,monaco,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,3,indianapolis 500,1950-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,4,swiss grand prix,1950-06-04,bremgarten,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,5,belgian grand prix,1950-06-18,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,6,french grand prix,1950-07-02,reims,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,7,italian grand prix,1950-09-03,monza,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,1,swiss grand prix,1951-05-27,bremgarten,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,2,indianapolis 500,1951-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,3,belgian grand prix,1951-06-17,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01


####Step 3,4: Standardizing Column names

 - Standardize column names using snake_case
 - Rename column names to make them meaningful

In [0]:
races_renamed_df = (
    races_selected_df
        .withColumnsRenamed({
            "circuitId": "circuit_id",
            "raceName": "race_name",
            "date": "race_date"})
)

In [0]:
display(races_renamed_df)

season,round,race_name,race_date,circuit_id,ingestion_timestamp,source_file,batch_id
1950,1,british grand prix,1950-05-13,silverstone,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,2,monaco grand prix,1950-05-21,monaco,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,3,indianapolis 500,1950-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,4,swiss grand prix,1950-06-04,bremgarten,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,5,belgian grand prix,1950-06-18,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,6,french grand prix,1950-07-02,reims,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,7,italian grand prix,1950-09-03,monza,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,1,swiss grand prix,1951-05-27,bremgarten,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,2,indianapolis 500,1951-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,3,belgian grand prix,1951-06-17,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01


####Step 5: Remove duplicate records

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season", "round"])

In [0]:
display(races_distinct_df)

season,round,race_name,race_date,circuit_id,ingestion_timestamp,source_file,batch_id
1950,1,british grand prix,1950-05-13,silverstone,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,2,monaco grand prix,1950-05-21,monaco,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,3,indianapolis 500,1950-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,4,swiss grand prix,1950-06-04,bremgarten,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,5,belgian grand prix,1950-06-18,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,6,french grand prix,1950-07-02,reims,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,7,italian grand prix,1950-09-03,monza,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,1,swiss grand prix,1951-05-27,bremgarten,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,2,indianapolis 500,1951-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,3,belgian grand prix,1951-06-17,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01


####Step 6: Transform the values of circuit_name to Title Case

In [0]:
races_final_df = (
    races_distinct_df
        .withColumn("race_name", F.initcap(F.col("race_name")))
)

display(races_final_df)

season,round,race_name,race_date,circuit_id,ingestion_timestamp,source_file,batch_id
1950,1,British Grand Prix,1950-05-13,silverstone,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,2,Monaco Grand Prix,1950-05-21,monaco,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,3,Indianapolis 500,1950-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,4,Swiss Grand Prix,1950-06-04,bremgarten,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,5,Belgian Grand Prix,1950-06-18,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,6,French Grand Prix,1950-07-02,reims,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1950,7,Italian Grand Prix,1950-09-03,monza,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,1,Swiss Grand Prix,1951-05-27,bremgarten,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,2,Indianapolis 500,1951-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01
1951,3,Belgian Grand Prix,1951-06-17,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01


####Step 7: Write the transformed data to the silver 'races' table

In [0]:
write_to_silver(
    input_df = races_final_df,
    target_table = silver_table,
    merge_condition = "t.season = s.season AND t.round = s.round",
    columns_to_update = [
        "season",
        "round",
        "race_name",
        "race_date",
        "circuit_id",
        "ingestion_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
display(spark.table(silver_table))

season,round,race_name,race_date,circuit_id,ingestion_timestamp,source_file,batch_id,created_timestamp,updated_timestamp
1950,3,Indianapolis 500,1950-05-30,indianapolis,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01,2026-08-05T14:26:20.411Z,2026-08-05T14:26:43.356Z
1950,5,Belgian Grand Prix,1950-06-18,spa,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01,2026-08-05T14:26:20.411Z,2026-08-05T14:26:43.356Z
1951,8,Spanish Grand Prix,1951-10-28,pedralbes,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01,2026-08-05T14:26:20.411Z,2026-08-05T14:26:43.356Z
1952,4,French Grand Prix,1952-07-06,essarts,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01,2026-08-05T14:26:20.411Z,2026-08-05T14:26:43.356Z
1954,6,German Grand Prix,1954-08-01,nurburgring,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01,2026-08-05T14:26:20.411Z,2026-08-05T14:26:43.356Z
1957,2,Monaco Grand Prix,1957-05-19,monaco,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01,2026-08-05T14:26:20.411Z,2026-08-05T14:26:43.356Z
1957,8,Italian Grand Prix,1957-09-08,monza,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01,2026-08-05T14:26:20.411Z,2026-08-05T14:26:43.356Z
1961,7,Italian Grand Prix,1961-09-10,monza,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01,2026-08-05T14:26:20.411Z,2026-08-05T14:26:43.356Z
1962,8,United States Grand Prix,1962-10-07,watkins_glen,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01,2026-08-05T14:26:20.411Z,2026-08-05T14:26:43.356Z
1963,4,French Grand Prix,1963-06-30,reims,2026-08-02T16:44:56.944Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/races.csv,2025-01,2026-08-05T14:26:20.411Z,2026-08-05T14:26:43.356Z
